# Bike-sharing network design — small instance demo

This notebook runs the optimisation model in [`network-design-bss/src/`](../network-design-bss/src) end to end
on a small instance: a **1.5 km-radius area of central Geneva**.

- Related research paper by [Zhenyu WU at INRIA](doc/Network_Design_BSS_PT_Zhenyu_0327.pdf)
- Dynamic demonstration map: <https://inria.github.io/inocs-sum-bss-pt-network-design>

## What the model does

The model integrates public transport structure and observed travel demand to decide **where to
place bike-sharing stations**, and **how many bikes each station holds in each time period**:

1. **Input pre-processing** — hex grid, public transport lines and stops, origin-destination demand
2. **Network building** — a multimodal graph over walk, bike and public transport arcs
3. **Decision model** — a mixed-integer program trading off investment budget, operating budget and
   multimodal accessibility, penalising suboptimal routes
4. **Output** — the selected stations, with capacity and bike inventory per period

## The demo instance

| | |
|---|---|
| Area | Geneva, 1.5 km radius around `6.15, 46.202778` — matching `RADIUS = 1.5` in `util/util.py` |
| Grid | 59 H3 cells at resolution 9 (~0.1 km² each); 3 dropped as unbuildable → **56 zones** |
| Public transport | 80 itineraries → **26 distinct in-bounds routes**, 82 in-bounds stop nodes |
| Candidate stations | **100**, after merging stops closer than 100 m |
| Demand | 1 658 observed bike trips → **1 453 inter-zone trips over 703 OD pairs**, split across 3 time periods |
| Budget | 80 000 total, 2.5 % operational |

The resulting mixed-integer program has **53 520 variables and 36 459 constraints**, so it needs a
real Gurobi licence — the size-limited licence bundled with the `gurobipy` wheel caps out at 2000 of
each, more than an order of magnitude short. [Academic licences are free](https://www.gurobi.com/academia/).

The **first** run takes about **50 minutes**, almost all of it enumerating the k shortest paths for
every OD pair — roughly 50 s per origin zone, twice (once for the no-bike-sharing baseline, once
with candidate stations). The OSM street networks and both path sets are cached under `osm_cache/`
and `data/shortest_paths_result/`, so every later run finishes in seconds.

Note that this only works because of the third runtime patch described in the notes: without it,
building the network arcs alone does not finish.

## 1. Environment

The model needs a geospatial and optimisation stack that is **not** the same as the packaged
`shared_mobility_network_optimizer` wheel used by the older demo. Install it with:

```
python -m pip install -r network-design-bss/src/requirements.txt
```

The package in `network-design-bss/src/` is never modified — see the notes at the end. The cell below
goes through [`demo/compat.py`](demo/compat.py), which prepares the process to run it.

Gurobi is included, but the licence bundled with the wheel is size-limited and **too small for
this instance**. Install an academic or full licence as `~/gurobi.lic` first.

In [1]:
import pathlib, sys

# Tolerate re-running this cell after bootstrap() has already moved us.
REPO = pathlib.Path.cwd()
while not (REPO / "network-design-bss" / "sdk-builder").is_dir():
    if REPO.parent == REPO:
        raise FileNotFoundError("repository root not found above " + str(pathlib.Path.cwd()))
    REPO = REPO.parent
SDK = REPO / "network-design-bss" / "sdk-builder"
if str(SDK) not in sys.path:
    sys.path.insert(0, str(SDK))
from sum_network_design_bss.compat import bootstrap

# Puts the frozen package on sys.path, moves into it (several of its paths are
# resolved against the working directory), creates the output directories it
# assumes exist, and installs the four runtime patches described in the notes.
# Must run before importing anything from the package.
PACKAGE = bootstrap()

print("working directory:", pathlib.Path.cwd())

working directory: /Users/rebecamurillo/Documents/INRIA/development/wp2-network-design-bss/inocs-sum-bss-pt-network-design/network-design-bss


## 2. Input data

The inputs live in `network-design-bss/src/data/geojson/<h3_version>/` and are produced by the companion
[SUM GTFS-to-GeoJSON](https://github.com/INRIA/inocs-sum-gtfs-geojson) package:

| File | Contents |
|---|---|
| `grid.geojson` | H3 hex cells covering the study area — the model's zones |
| `stops.geojson` | public transport stops (GTFS) |
| `itineraries.geojson` | public transport line geometries |
| `bike_trips.geojson` | observed bike trips, used to derive demand |
| `od.csv` | origin-destination demand, derived from `bike_trips.geojson` |

`od.csv` is generated rather than shipped by the data package. It is built by
[`demo/od_builder.py`](demo/od_builder.py) — which lives outside the frozen package — either with
`python demo/od_builder.py` or from here:

In [2]:
from util.util import h3_version, H3_resolution
from sum_network_design_bss.od_builder import build_od_from_trips

GEOJSON_DIR = PACKAGE / "data" / "geojson" / h3_version
print("h3_version:", h3_version, "| resolution:", H3_resolution)
print("files:", sorted(p.name for p in GEOJSON_DIR.iterdir()))

build_od_from_trips(GEOJSON_DIR)

h3_version: geneva_1.5km-radius | resolution: 9
files: ['bike_trips.geojson', 'grid.geojson', 'itineraries.geojson', 'od.csv', 'stops.geojson']
✅ Saved: /Users/rebecamurillo/Documents/INRIA/development/wp2-network-design-bss/inocs-sum-bss-pt-network-design/network-design-bss/data/geojson/geneva_1.5km-radius/od.csv
   1658 trips read, 9 with an endpoint outside the grid, 1453 counted over 703 OD pairs


'/Users/rebecamurillo/Documents/INRIA/development/wp2-network-design-bss/inocs-sum-bss-pt-network-design/network-design-bss/data/geojson/geneva_1.5km-radius/od.csv'

In [3]:
import pandas as pd

od = pd.read_csv(GEOJSON_DIR / "od.csv")
print(f"{od['flow'].sum()} trips over {len(od)} OD pairs")
od.head()

1453 trips over 703 OD pairs


,origin_cell,dest_cell,flow
0,897a8dcac53ffff,897a8dc120fffff,24
1,897a8dcac5bffff,897a8dc120bffff,22
2,897a8dc120bffff,897a8dcac5bffff,21
3,897a8dcaccfffff,897a8dc127bffff,19
4,897a8dca1b3ffff,897a8dc1223ffff,18


## 3. Build the instance

`generate_h3_instances()` assembles a scenario — grid, public transport, candidate stations and
demand split into time periods — and serialises it to `h3_instances_json/`.

The demand split is stochastic (multinomial over the period weights) but seeded, so the instance is
reproducible.

In [4]:
from instance_builder import generate_h3_instances

instance = generate_h3_instances()

print("\nconfig name       :", instance.config_name)
print("grid zones        :", len(instance.grid_generator.grid_centers))
print("public transport  :", len(instance.public_transport.routes), "routes")
print("candidate stations:", len(instance.all_stations))
print("demand entries    :", len(instance.od_demand),
      "| total flow:", sum(instance.od_demand.values()))

[dedupe] route PT_2 duplicates PT_19, skipped.
[dedupe] route PT_G+ duplicates PT_E+, skipped.
[dedupe] route PT_G duplicates PT_E, skipped.
✅ Saved: h3_instances_json/Geneva_H3True_SEED20_DISmultinomial_P0.40-0.18-0.42_BUD80000_OP0.0250.txt

config name       : Geneva_H3True_SEED20_DISmultinomial_P0.40-0.18-0.42_BUD80000_OP0.0250
grid zones        : 56
public transport  : 26 routes
candidate stations: 100
demand entries    : 971 | total flow: 1453


## 4. Build the network and solve the shortest paths

`get_instance_attribute()` reloads the serialised instance and plots the grid; then
`get_shortest_path_solver()` builds the multimodal graph and enumerates the k shortest paths for
every OD pair, both **before** bike-sharing (the baseline) and **after** (with candidate stations).

> **This is the slow step** — it routes over the OSM street network. Results are cached under
> `data/shortest_paths_result/`, so re-running the notebook is much faster than the first pass.

In [5]:
from input_handler.instance_attribute_extracter import get_instance_attribute
from main import get_shortest_path_solver

config_name, demand_generator, grid_generator, public_transport, station_layout = \
    get_instance_attribute(instance)

network_with_bss, shortest_path_solver = get_shortest_path_solver(
    grid_generator, public_transport, station_layout)

print("network nodes:", network_with_bss.graph.number_of_nodes())
print("network arcs :", network_with_bss.graph.number_of_edges())

The total demand is 1453.
Period 0: total demand = 585, active OD pairs = 392, demand per OD = 1.49
Period 1: total demand = 245, active OD pairs = 192, demand per OD = 1.28
Period 2: total demand = 623, active OD pairs = 387, demand per OD = 1.61


[plot] Figure saved to: /Users/rebecamurillo/Documents/INRIA/development/wp2-network-design-bss/inocs-sum-bss-pt-network-design/network-design-bss/plot/H3_Grid_with_Candidate_Stations.png


[plot] PT stops figure saved to: /Users/rebecamurillo/Documents/INRIA/development/wp2-network-design-bss/inocs-sum-bss-pt-network-design/network-design-bss/plot/H3_Grid_with_PT_Stops.png


[SP cache] Loaded 12248 entries
Grid center walks to userOD processed: 1540 arcs added.

Grid center walks to publicTransportStop processed: 4592 arcs added.

Node: 
  - userOD: 56
  - publicTransportStop: 82

 Arc:
  - Walk: 18906
  - PT: 186
[SP cache] Saved 12248 entries
✅ Found file 'shortest_paths_cache_size7_k1_['real_pt_lines_radius2']_before.pkl', loading...


[SP cache] Loaded 12248 entries
Grid center walks to userOD processed: 0 arcs added.

Grid center walks to bikeStation processed: 193 arcs added.

Grid center walks to publicTransportStop processed: 110 arcs added.

PT stop walks to bike stations processed: 426 arcs added.

Biking processed: 2058 arcs added.

Node: 
  - userOD: 56
  - bikeStation: 100
  - publicTransportStop: 38

 Arc:
  - Walk: 1574
  - Bike: 4116
  - PT: 186
[SP cache] Saved 12248 entries
✅ Found file 'shortest_paths_cache_size7_k3_['real_pt_lines_radius2']_after.pkl', loading...
network nodes: 194
network arcs : 5876


## 5. Solve the optimisation model

`solve_mode="integrated"` solves the full mixed-integer program in one shot — the exact optimum.

The other modes in `optimization_model_solver` are `"sequential"` (two-stage: design, then
operations on the fixed design), `"benders"` and `"alns"`. **`"benders"` and `"alns"` are not
usable in this repository** — see the closing notes.

In [6]:
from model.sequential_optimization import optimization_model_solver
from util.util import EPSILON

bike_sharing_model = optimization_model_solver(
    demand_generator,
    EPSILON,                 # penalty weight on dispatch cost
    network_with_bss,
    shortest_path_solver,
    solve_mode="integrated",
)

gurobi_model = bike_sharing_model.model
print("\nstatus     :", gurobi_model.Status, "(2 = optimal)")
print("objective  :", gurobi_model.ObjVal)
print("variables  :", gurobi_model.NumVars)
print("constraints:", gurobi_model.NumConstrs)

Set parameter Username


Set parameter LicenseID to value 2790753


Academic license - for non-commercial use only - expires 2027-03-12


=== Sets Initialization Completed ===
Total OD pairs (K): 3067
Total Bike Stations (B): 100
=== Parameters Initialization Completed ===
Start setting variables


=== Variables Initialization Completed ===
=== Objectives Initialization Completed ===


=== Constraints Initialization Completed ===
Final total constraints count: 0
Set parameter TimeLimit to value 3600


Set parameter MIPGap to value 0


🔍 TimeLimit parameter: 3600.0
Set parameter Method to value 3


---------------------------
Statistics for model 'NetworkDesign':
  Problem type                : MIP
  Linear constraint matrix    : 36459 rows, 53520 columns, 145507 nonzeros
  Variable types              : 0 continuous, 53520 integer (1864 binary)
  Matrix range                : [1e+00, 1e+02]
  Objective range             : [7e-01, 3e+00]
  Bounds range                : [1e+00, 3e+01]
  RHS range                   : [1e+00, 8e+04]
✅ Gurobi 结果已保存到 gurobi_results_20260902_193850.json
✅ GeoJSON saved -> /Users/rebecamurillo/Documents/INRIA/development/wp2-network-design-bss/inocs-sum-bss-pt-network-design/network-design-bss/data/output/geojson/selected_bike_stations.geojson_2026-09-02_19-38-50.geojson (features=81)

✅ 选定的站点（y=1）及其初始库存 v 和容量 z：
🚲 站点 Node(BS-897a8dcac8fffff, coordinate=(6.1535556998967325, 46.19638405213572), is_origin=False, : y = 1.0, v(initial inventory) = 3.0, z(capacity) = 7.0
🚲 站点 Node(BS-897a8dc1237ffff, coordinate=(6.1407875761988455, 46.198121937911054), is_ori

## 6. Results

`ExperimentHandler` computes the reporting metrics and appends a row to
`experiment_results.csv`: how many stations were selected, their capacities and fill ratios, the
share of OD demand covered, travel time saved against the no-bike-sharing baseline, and the
rebalancing effort.

In [7]:
from output_handler.experiment import ExperimentHandler

# Constructing it is enough: __init__ calls record() and save() itself
# (experiment.py:39), so calling them again would append the same row to
# experiment_results.csv twice. record() computes the metrics and sets
# last_row (experiment.py:69); save() flushes the row and clears the buffer,
# but last_row survives it.
experiment = ExperimentHandler(bike_sharing_model, network_with_bss, config_name)
summary = experiment.last_row
pd.Series({k: summary[k] for k in [
    "n_candidates", "n_selected_stations", "n_reg_station", "n_trans_station",
    "avg_capacity_reg", "avg_capacity_trans",
    "covered_od_ratio", "total_time_gain", "average_time_gain",
    "flow_bike_only", "flow_bike_pt", "obj_val",
] if k in summary}).to_frame("value")

,value
n_candidates,100.000000
n_selected_stations,81.000000
n_reg_station,52.000000
n_trans_station,29.000000
avg_capacity_reg,14.538462
avg_capacity_trans,11.827586
covered_od_ratio,0.935000
total_time_gain,7310.973232
average_time_gain,5.628155
flow_bike_only,782.000000


In [8]:
# Which candidate stations were selected, and at what capacity
design = bike_sharing_model.get_design_solution()
selected = {i: design["w"][i] for i, built in design["y"].items() if built > 0.5}

print(f"{len(selected)} of {len(design['y'])} candidate stations selected\n")
for station, capacity in sorted(selected.items(), key=lambda kv: -kv[1]):
    print(f"  {getattr(station, 'node_id', station):<28} capacity {capacity:6.1f}")

81 of 100 candidate stations selected

  BS-897a8dc120bffff           capacity   30.0
  BS-897a8dca1a3ffff           capacity   30.0
  BS-897a8dc1277ffff           capacity   30.0
  BS-897a8dc1247ffff           capacity   30.0
  BS-897a8dcacd7ffff           capacity   30.0
  BS-897a8dcac53ffff           capacity   30.0
  transfer_PT_12-4             capacity   30.0
  transfer_PT_12-5             capacity   30.0
  transfer_PT_25-2             capacity   30.0
  transfer_PT_25-3             capacity   30.0
  transfer_PT_25-4             capacity   30.0
  BS-897a8dcaccfffff           capacity   29.0
  BS-897a8dc123bffff           capacity   28.0
  BS-897a8dc1273ffff           capacity   28.0
  BS-897a8dc126fffff           capacity   27.0
  BS-897a8dc120fffff           capacity   27.0
  BS-897a8dc1223ffff           capacity   25.0
  BS-897a8dc127bffff           capacity   24.0
  BS-897a8dc1267ffff           capacity   21.0
  BS-897a8dcacc7ffff           capacity   19.0
  BS-897a8dcaccbffff 

### Reference values

Running this notebook on the committed input files reproduces:

| metric | value | meaning |
|---|---|---|
| Gurobi status | `2` (optimal) | proved optimal; `MIPGap` is set to 0 |
| model size | 53 520 vars / 36 459 constrs | 3 067 OD-path pairs over 100 candidates |
| `obj_val` | 1128.3558538 | weighted objective — **stable to 13 significant digits across runs** |
| `n_selected_stations` | 81 of 100 | the 80 000 budget **binds** — 19 candidates go unfunded |
| `n_reg_station` / `n_trans_station` | 52 / 29 | zone-centre stations vs stations co-located with a PT stop |
| `avg_capacity_reg` / `avg_capacity_trans` | ≈14.5 / ≈11.8 † | well below `CAPACITY_UB`, so capacity is being traded off too |
| `covered_od_ratio` | ≈0.933 † | share of OD demand served by a bike leg |
| `total_time_gain` | ≈7 311 min † | saved against the no-bike-sharing baseline |
| `average_time_gain` | ≈5.62 min † | per covered trip |
| `flow_bike_only` / `flow_bike_pt` | ≈784 / ≈515 † | bike-only vs bike+public-transport trips |

† **These vary between runs; the marked values are one representative optimal solution.**

The model has alternative optima. Across **eight** runs of this same instance the objective agreed
to 12 significant digits and the design was identical every time — 81 stations selected, 52 regular
and 29 transfer — while the derived quantities moved in their third or fourth significant figure:

| | observed across 8 runs | spread |
|---|---|---|
| `obj_val` | 1128.35585382373… | none to 12 s.f. |
| `n_selected_stations`, `n_reg_station`, `n_trans_station` | 81, 52, 29 | none |
| `covered_od_ratio` | 0.930 – 0.936 | 0.6 % |
| `total_time_gain` | 7 305.0 – 7 317.7 min | 0.17 % |
| `average_time_gain` | 5.619 – 5.633 min | 0.25 % |
| `avg_capacity_reg` / `avg_capacity_trans` | 14.46 – 14.56 / 11.76 – 11.90 | 0.7 % / 1.2 % |
| `flow_bike_only` / `flow_bike_pt` | 782 – 787 / 512 – 517 (total 1 299 – 1 300) | 0.6 % |

**These are observed spreads, not bounds.** Runs 5, 6 and 8 each landed outside the range the
earlier ones had established. What does not move is the objective and the design.

The cause is in the frozen code: `model/model_template.py:81` sets `Method = 3`, commented as
*"Network Simplex"*. In Gurobi that value selects **concurrent** optimisation, which is documented
as non-deterministic — `Method = 4` is the deterministic variant — and no `Seed` is fixed. So the
solver is free to return a different member of the optimal face each time.

**When comparing runs, compare the objective, not the derived metrics.**

Timings on the reference machine: instance 1.5 s, attributes 4 s, network and shortest paths
**2 861 s cold / 7–11 s warm**, solve 166–169 s.

Sanity checks worth keeping an eye on, because they are what caught the edge-length bug described
in §8. Measured on this instance:

| arcs | count | median length | max length | median time |
|---|---|---|---|---|
| Walk | 1 574 | 0.239 km | 0.779 km | 3.59 min (≈4 km/h) |
| Bike | 4 116 | 0.687 km | 1.729 km | 3.06 min (20 km/h + 1 min access/egress) |
| PT (all lines) | 186 | ≈0.3 km | 0.72 km | 1–2 min (scheduled, not derived from a speed) |

Distances in the thousands of kilometres, or a `total_time_gain` in the millions, mean the street
network was measured in projected units — delete `osm_cache/` and `data/shortest_paths_result/`
and re-run.

Unlike the earlier 1 km instance, where every candidate was funded, the budget binds here: 19 of
100 candidates are left unbuilt and average capacities sit at roughly half of `CAPACITY_UB`. The
siting and sizing trade-off the model is designed to express is therefore active. Lowering `BUDGET`
in `util/util.py` tightens it further.

The run also writes, relative to `network-design-bss/src/`:

- `plot/` — the grid, candidate stations and public transport stop maps
- `data/output/<version>/` — GeoJSON of the selected stations, solver logs and result JSON
- `experiment_results.csv` — one row per solve, appended

The selected-stations GeoJSON is what feeds the
[dynamic demonstration map](https://inria.github.io/inocs-sum-bss-pt-network-design).

## 7. Configuration

Model parameters are module-level constants in [`util/util.py`](../network-design-bss/src/util/util.py),
read at import time. Change them **before** importing the model modules.

| Parameter | Default | Description |
|---|---|---|
| `h3_version` | `"geneva_1.5km-radius"` | subdirectory of `data/geojson/` holding the inputs |
| `H3_resolution` | `9` | must match the resolution of the cells in `grid.geojson` |
| `NUM_SHORTEST_PATHS` | `3` | number of shortest paths enumerated per OD pair |
| `OD_COVERAGE_RATIO` | `0.5` | minimum share of OD pairs that must be serviced |
| `WALK_CATCHMENT_RADIUS` | `0.3` km | walking catchment of a station |
| `RIDE_CATCHMENT_RADIUS` | `1` km | cycling catchment of a station |
| `PT_TRANSFER_RADIUS` | `0.5` km | maximum distance for a public transport transfer |
| `WALK_SPEED` / `RIDE_SPEED` | `4` / `20` km/h | modal speeds |
| `TIME_PERIODS` | `3` | number of demand periods |
| `PENALTY_COEFFICIENT` | `0.1` | penalty for choosing a suboptimal path |
| `CAPACITY_UB` | `30` | maximum station capacity, in bikes |
| `MIN_CAPACITY_IF_BUILT` | `5` | minimum capacity of a station that is built |
| `EPSILON` | `0.04` | penalty weight on dispatch cost |
| `REBALANCING_FLAG` | `True` | whether rebalancing between periods is modelled |

Budget and demand-split parameters are per-scenario instead, set through `ScenarioConfig` in
[`scenario/scenario_config.py`](../network-design-bss/src/scenario/scenario_config.py) — see
`instance_builder.generate_h3_instances()` for how they are assembled.

## Notes and known gaps

**The model package is frozen.** [`network-design-bss/src/`](../network-design-bss/src) is Zhenyu WU's source,
imported with its full history and topped by the commit *"Submitted version"*. **No `.py` file in it
is modified by this repository** — results stay attributable to the code as submitted. Everything
needed to run it lives in [`network-design-bss/sdk-builder/`](../network-design-bss/sdk-builder) instead, and `git diff HEAD -- 'network-design-bss/src/**/*.py'`
should always be empty. What that costs is four runtime patches, applied by `demo/compat.py` before
the package is imported:

- **`model.benders_optimization` stub.** `model/sequential_optimization.py:2` imports
  `model/benders_optimization.py` at module level, but that file was never committed. Without the
  stub the module fails to import, which takes `"integrated"` and `"sequential"` down with it even
  though neither uses Benders decomposition. Registering the name in `sys.modules` satisfies the
  import; constructing the class still raises, so `"benders"` fails loudly.
- **`add_edge_lengths` guard.** `network/osmnx_network.py:48` calls `add_edge_lengths()` *after*
  `project_graph()`. That function reads each node's `x`/`y` as lon/lat degrees, so on a projected
  graph (UTM eastings/northings, e.g. `280015, 5122887`) it inflates every edge by roughly
  **86 000x** — the median edge going from 17.6 m to 1 512 km — and overwrites the correct lengths
  `graph_from_point` supplied. Walk and bike arcs then cost astronomically more than public
  transport arcs, which keep correct times in minutes. The guard returns projected graphs untouched
  when they already carry lengths. **Any result produced without it is invalid.**

- **Undirected-graph memoisation.** `network/osmnx_network.py:97` rebuilds an undirected copy of
  the whole street network inside `shortest_path_km_min`, on every cache miss. On the Geneva walk
  graph (25 399 nodes, 72 920 edges) that copy costs **0.809 s** against **0.027 s** for the
  Dijkstra that follows, so roughly **97%** of shortest-path time went on reproducing a graph that
  never changes. The graphs are only read after `NetworkBuilder.__init__` returns, so the copy is
  memoised per graph. Distances and travel times are bit-identical; only the redundant work goes.
  Without this the instance below does not finish in a usable time.

- **Solver stdout restore.** `model/model_template.py:82-84` routes the Gurobi log to a file with
  `sys.stdout = log_file`, then restores with `sys.stdout = sys.__stdout__`. In a plain interpreter
  those are the same object. In a Jupyter kernel they are not — `sys.stdout` is ipykernel's capture
  stream and `sys.__stdout__` is the real process stdout — so the restore silently redirects **the
  rest of the session** away from the notebook and into the terminal that launched the kernel.
  Without the patch, sections 5 and 6 below execute correctly but display nothing. The patch saves
  the stream that was actually in place and restores it in a `finally`.

`alns/alns.py`, `alns/verify.py` and `output_handler/comparison_writer.py` are also missing, so
`"alns"` mode and `main_run_alns` / `main_verify_alns` / `main_compare_methods` raise
`ModuleNotFoundError`. They are imported lazily, inside the functions that use them, so they block
nothing else.

**Hex cell identifiers are transposed — deliberately left that way.** The data package builds the
grid by handing `h3.LatLngPoly` a ring of `(lon, lat)` pairs when it wants `(lat, lng)`, so the cell
`id` values describe a location off the Somali coast rather than Geneva. The exported *geometry* is
correct, because the same swap is undone when the boundary is written back out.

This is not corrected, because the frozen model makes the identical swap:
`input_handler/pt_generator.py:115` calls `h3.latlng_to_cell(lon, lat, H3_resolution)`. Both sides
therefore live in the same transposed id-space and agree with each other, and
`input_handler/h3_grid_generator.py:42` already un-swaps the centre with a comment saying so.
Fixing only the data package would desynchronise the ids from code that cannot be changed. Treat
cell ids as opaque zone keys and never reconstruct one from coordinates — `demo/od_builder.py`
assigns trips to zones by point-in-polygon test for exactly this reason.

**Public transport edge times are supplied but mis-indexed by the model.** `itineraries.geojson`
now carries `edge_travel_time_seconds`, read from the GTFS schedule as the span from one stop's
departure to the next stop's arrival. The frozen `input_handler/pt_generator.py` reads that array at
line 83, *then* filters the route's coordinates to those inside the grid polygon, then indexes the
array by the **filtered** position at line 137. Any route clipped at the study-area boundary
therefore gets travel times shifted by the number of dropped leading stops. The values stay in a
plausible range, so this degrades realism rather than breaking the run — but it is a real defect in
the frozen code, not in the data.

**Scaling up.** `h3_version` also accepts the `geneva_5km-radius` and `geneva_10km-radius` datasets
if you generate them from the data package. Both need a full Gurobi licence, and shortest-path
enumeration grows quickly with the number of OD pairs.